# 01 · Coleta → camada bronze

**MVP — Onde subsidiar o frete no e-commerce brasileiro: pipeline de dados em nuvem (Databricks)**

Universidade de Brasília (UnB) · Departamento de Engenharia de Produção · Sistemas de Suporte à Decisão
Prof. André Luiz Marques Serrano · **Aluna: Brunna Gavião Moraes — 231025341**


Este notebook cria a estrutura na nuvem e coleta os dados **sem alterá-los**:

1. cria no Unity Catalog os schemas `frete_bronze`, `frete_silver`, `frete_gold` e o Volume `frete_bronze.arquivos`;
2. baixa os 9 arquivos da Olist (Kaggle) e a população por UF do IBGE (API SIDRA);
3. grava tudo na camada bronze, em uma pasta datada (`coleta=AAAA-MM-DD`), com o hash SHA-256 de cada arquivo.

| Fonte | Conteúdo | Licença | Acesso |
|---|---|---|---|
| [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) | ~100 mil pedidos (2016–2018), 9 CSVs | CC BY-NC-SA 4.0 | API pública do Kaggle (`kagglehub`) |
| [IBGE — SIDRA, tabela 6579](https://sidra.ibge.gov.br/tabela/6579) | População residente estimada por UF, 2018 | Dados públicos (Lei de Acesso à Informação) | API SIDRA |

In [ ]:
%pip install -q kagglehub statsmodels

In [ ]:
# Configuração comum: localiza o repositório, importa o módulo de apoio e detecta a plataforma.
# No Databricks a variável `spark` já existe; fora dele o mesmo código roda localmente (Parquet + DuckDB).
import os, sys, importlib
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
import mvp_frete as mvp
importlib.reload(mvp)

import pandas as pd
pd.set_option("display.max_columns", 50, "display.width", 200)

plat = mvp.Plataforma(spark if "spark" in globals() else None)
print("Ambiente :", plat.preparar())
print("Bronze   :", plat.dir_bronze)
print("Evidências:", plat.dir_evidencias)

## 1. Pasta da coleta

Cada execução grava em uma pasta própria, datada. Assim a coleta fica rastreável e uma nova coleta não apaga a anterior.

In [ ]:
import datetime as dt, shutil, json

data_coleta = dt.date.today().isoformat()
dir_coleta = plat.dir_bronze / f"coleta={data_coleta}"
dir_coleta.mkdir(parents=True, exist_ok=True)
dir_upload = plat.dir_bronze / "upload_manual"   # plano B: arquivos enviados pela interface
print(dir_coleta)

## 2. Olist (Kaggle)

O download usa a API pública do Kaggle, sem login. **Plano B:** se o ambiente bloquear o acesso à internet, envie os 9 CSVs
pela interface (*Catalog → workspace → frete_bronze → arquivos → Upload to this volume*) para a pasta `upload_manual/`
e execute a célula de novo.

In [ ]:
origem = None
try:
    import kagglehub
    origem = Path(kagglehub.dataset_download(mvp.KAGGLE_DATASET))
    forma = f"kagglehub ({mvp.KAGGLE_DATASET}, pasta {origem.name})"
except Exception as erro:
    print("Download pelo Kaggle indisponível:", type(erro).__name__, erro)
    if all((dir_upload / a).exists() for a in mvp.ARQUIVOS_OLIST):
        origem, forma = dir_upload, "upload manual pela interface do Databricks"
    else:
        raise RuntimeError(f"Envie os 9 CSVs da Olist para {dir_upload} e execute novamente.")

for arq in mvp.ARQUIVOS_OLIST:
    shutil.copy(origem / arq, dir_coleta / arq)
print("Olist copiada para a bronze via", forma)

## 3. IBGE — população residente estimada por UF (2018)

In [ ]:
try:
    mvp.baixar_ibge_populacao(dir_coleta / mvp.ARQUIVO_IBGE)
    forma_ibge = f"API SIDRA ({mvp.IBGE_URL})"
except Exception as erro:
    print("API do IBGE indisponível:", erro, "→ usando a cópia versionada em dados/")
    shutil.copy(RAIZ / "dados" / mvp.ARQUIVO_IBGE, dir_coleta / mvp.ARQUIVO_IBGE)
    forma_ibge = "cópia versionada no repositório (dados/ibge_populacao_uf_2018.csv)"
pd.read_csv(dir_coleta / mvp.ARQUIVO_IBGE).head()

## 4. Metadados de reprodutibilidade

Data da coleta, forma de acesso, volume (linhas, colunas, MB) e **hash SHA-256** de cada arquivo. Quem repetir a
coleta pode comparar os hashes e confirmar que recebeu exatamente os mesmos arquivos.

In [ ]:
meta = [mvp.metadados_arquivo(dir_coleta / a, "Kaggle — Olist") for a in mvp.ARQUIVOS_OLIST]
meta.append(mvp.metadados_arquivo(dir_coleta / mvp.ARQUIVO_IBGE, "IBGE — SIDRA 6579"))
metadados = {
    "data_coleta": dt.datetime.now().isoformat(timespec="seconds"),
    "ambiente": plat.ambiente(),
    "pasta_bronze": str(dir_coleta),
    "forma_coleta_olist": forma,
    "forma_coleta_ibge": forma_ibge,
    "licenca_olist": "CC BY-NC-SA 4.0",
    "arquivos": meta,
}
(dir_coleta / "_metadados_coleta.json").write_text(json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8")
(plat.dir_evidencias / "metadados").mkdir(parents=True, exist_ok=True)
(plat.dir_evidencias / "metadados" / "_metadados_coleta.json").write_text(
    json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8")

tab_meta = pd.DataFrame(meta)
print(f"{len(tab_meta)} arquivos · {tab_meta.linhas.sum():,} linhas · {tab_meta.mb.sum():.2f} MB".replace(",", "."))
display(tab_meta[["arquivo", "fonte", "linhas", "colunas", "mb", "sha256"]])

In [ ]:
# Conferência: o que está gravado na nuvem (listagem da pasta da coleta na bronze)
display(pd.DataFrame([{"arquivo": p.name, "bytes": p.stat().st_size} for p in sorted(dir_coleta.iterdir())]))

> **Evidência a capturar:** *Catalog Explorer → workspace → frete_bronze → Volumes → arquivos → coleta=…* mostrando os
> 10 arquivos e o `_metadados_coleta.json` (salvar como `evidencias/01_volume_bronze.png`).